### Autonomous Traders

An equity trading simulation with four Traders and a Researcher, powered by a set of MCP servers and their tools and resources:

1. An Accounts server.
2. Push notifications on phone, to alert when something happens
3. Market data, for live or simulated share prices
4. Fetch, to read a web page through a local headless browser
5. Tavily, for web search
6. Memory, a knowledge graph the researcher writes to and reads back

Plus resources to read each trader's account and their investment strategy.


### The architecture

Here is how the pieces fit together. Each of the four traders is an agent with its own MCP servers for accounts, push notifications and market data, and it calls a researcher agent as a tool. The researcher is itself an agent, with its own MCP servers for fetching pages, web search and memory. Orange is an agent, blue is an MCP server, and there are six MCP servers in all.

While I've given the Agents names like "Trader" and "Researcher", my motivation for this architecture was managing the context effectively with the right prompts and tools for reliable outcomes. It's key to avoid the trap of assigning Agent responsibilities just because that's how human teams are organized..

<img src="./assets/architecture.png" width="820" />

In [10]:
from dotenv import load_dotenv, dotenv_values
from IPython.display import Markdown, display
import os

from backend.accounts import Account
from backend.mcp_servers import trader_mcp_servers, researcher_mcp_servers
from contextlib import AsyncExitStack
from agents import Runner, trace, add_trace_processor


from backend.market import get_share_price
from backend.accounts import Account
from backend.accounts_client import read_accounts_resource
from backend.reset import reset_traders
from backend.mcp_servers import trader_mcp_servers, researcher_mcp_servers
from backend.traders import get_researcher, get_researcher_tool, Trader
from backend.tracers import LogTracer
import json


In [2]:
load_dotenv(override=True)

True

In [3]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
POLYGON_API_KEY = os.getenv("POLYGON_API_KEY")

In [4]:
warren = Account.get("Warren")
print("Balance:", warren.balance)
print("Strategy:", warren.get_strategy())

Balance: 10000.0
Strategy: 


### The MCP servers, and their tools

The trader and the researcher each get their own set of MCP servers, and `mcp_servers.py` builds them. Let's start each one and count the tools they expose.

In [5]:
# Check how many MCP server 
# Get a LIST of servers list[MCPServerStdio]:
servers = trader_mcp_servers() + researcher_mcp_servers("Warren")

count = 0

for server in servers:
    async with server:
        tools = await server.list_tools()
        count += len(tools)
print(f"We have {len(servers)} MCP servers, and {count} tools") 

We have 6 MCP servers, and 17 tools


### The Researcher

The trader does not search the web itself. Instead it calls a Researcher agent as one of its tools. 

The researcher has its own MCP servers (3 of them): 

a) Fetch to read pages.

b) Tavily for web search.

c) Memory it writes to and reads back.

Tavily offers several tools, from plain search to a heavyweight deep-research mode. We restrict its server to `tavily_search`, which keeps the researcher fast and focused. 

**Choosing which tools an agent can see is itself context engineering.**

Wrapping an agent as a tool is different from a handoff: with a tool the trader stays in control and gets the researcher's answer back, where a handoff would pass the whole conversation over.

In [6]:
async with AsyncExitStack() as stack:
    servers = [await stack.enter_async_context(server) for server in researcher_mcp_servers("Warren")]
    researcher = await get_researcher(servers, "gpt-5.4-mini")
    with trace("Researcher"):
        result = await Runner.run(researcher, "What's the latest news on Amazon?", max_turns=30)
display(Markdown(result.final_output))

Latest Amazon news is mostly about **AI/cloud growth vs. heavy spending**.

- **Shares recently rose** on stronger AWS/AI sentiment and softer inflation.
- **Q2 2026 earnings are due July 30**; investors will focus on AWS growth, margins, and capex.
- **Q1 2026 was strong**: revenue **$181.5B** and operating income **$23.9B**.
- **Big theme:** Amazon is reportedly spending around **$200B in 2026** on data centers, networking, and AI infrastructure.
- **Recent headlines:**
  - AWS launched a **$1B forward-deployed AI engineering unit**
  - **Prime Day** was reported as a retail tailwind
  - Amazon faced a **FTC settlement** over Prime signups
  - Amazon’s Australian unit was **sued over Prime Video ads**

If you want, I can turn this into a **bullish/bearish investment read** on AMZN.

### Check your trace

https://platform.openai.com/traces

In [7]:
researcher_tool = await get_researcher_tool(researcher_mcp_servers("Warren"), "gpt-5.4-mini")
print("Tool name:", researcher_tool.name)
print("Description:", researcher_tool.description)

Tool name: Researcher
Description: This tool researches online for news and opportunities, either based on your specific request to look into a certain stock, or generally for notable financial news and opportunities. Describe what kind of research you're looking for.


## The Trader

The `Trader` class in `traders.py` brings it together. It builds the researcher as a tool, creates the trader agent over the trader's MCP servers (accounts, push and market data) with that tool, and runs it against a message built from the trader's strategy and current account.

There is one more piece worth seeing. The OpenAI Agents SDK lets you plug into its tracing, so you can follow what each agent does in code. `tracers.py` has a custom trace processor that records each trader's steps to the database, which is how we surface their thinking on the dashboard. We register it now, then run Warren.

In [8]:
add_trace_processor(LogTracer())

warren = Trader("Warren", "Patience", "gpt-5.4-mini")
await warren.run()

### How did Warren do?

Reading the account back through its MCP resource shows the trades Warren made and the state of the portfolio.

In [11]:
resources = await read_accounts_resource("Warren")
info = json.loads(resources)
print(info["transactions"][-1])

{'symbol': 'VOOG', 'quantity': 60, 'price': 82.07382, 'timestamp': '2026-07-17 16:52:57', 'rationale': 'Balanced growth leadership trade. Research highlighted VOOG as a lower-volatility way to stay with the market leaders, and the latest prior-day bar shows it holding near $81.91 close with tight range, fitting a measured momentum allocation within available cash.'}
